# Mini-DrawBench · Colab runner
**MSAI-630 Residency Hackathon, Project 2.** SDXL-Turbo vs SD 1.5, scored by CLIP and by blind human raters.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. No API keys needed.

Run the cells top to bottom. Every step is resumable: if Colab disconnects, re-run from the top and finished images, scores and ratings are skipped, because everything is saved to Google Drive.

| Step | Time on a T4 (approx.) |
|---|---|
| 1-3 Setup + smoke test | 5 min (first model download) |
| 4 Generate 384 images | 15-20 min |
| 5 CLIP scoring | 2 min |
| 6 Blind rating (the team) | 30-45 min with 5 raters |
| 7 Analysis | 1 min |
| 8 Diversity stretch goal | 20-25 min |

In [ ]:
# 1. Code + data locations. Data goes to Drive so a disconnect never loses work.
REPO_URL = "https://github.com/PatrickPhang/MSAI-630-Residency.git" 
DATA_DIR = "/content/drive/MyDrive/mini-drawbench-data"

from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["MDB_DATA"] = DATA_DIR
os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists("/content/mini-drawbench"):
    !git clone -q {REPO_URL} /content/mini-drawbench
%cd /content/mini-drawbench
!git pull -q

In [ ]:
# 2. Install + GPU check
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Smoke test: 2 prompts x 1 seed x 2 models. Look at the images before running everything.
!python generate.py --models sdxl_turbo sd15 --ids C01 T01 --seeds 0
from IPython.display import display
from PIL import Image
for m in ["sdxl_turbo", "sd15"]:
    for pid in ["C01", "T01"]:
        p = f"{DATA_DIR}/images/{m}/{pid}_s0.png"
        if os.path.exists(p):
            print(m, pid); display(Image.open(p).resize((256, 256)))

## 4. Generate the full benchmark
48 prompts × 4 seeds × 2 models = 384 images. Write and freeze `prompts.csv` **before** this step (the handout says to write the test set before seeing outputs).

In [ ]:
!python generate.py --models sdxl_turbo sd15

In [ ]:
# 5. CLIP alignment scores (OpenCLIP ViT-H-14; not the encoder either generator uses verbatim)
!python score_clip.py

## 6. Blind human rating
Run the cell, open the **public gradio.live link** it prints, and share it in the team chat. Everyone rates from their own laptop at the same time; the app hands out the least-rated images first, so you don't need to coordinate. Aim for every image rated by 2 people.

Rules for raters: be strict, rate what you see, don't discuss images while rating (that would inflate inter-rater agreement).

**Stop the cell** (■) when the progress line says 0 images under 2 ratings.

In [ ]:
!python rate_app.py --share

In [ ]:
# 7. Analysis: tables, charts, report.md with every headline number
!python analyze.py
from IPython.display import Markdown
import glob
for f in sorted(glob.glob(f"{DATA_DIR}/results/fig*.png")):
    print(os.path.basename(f)); display(Image.open(f))

## 8. Stretch goal: diversity / mode collapse
10 samples of 5 prompts per model, plus an SD 1.5 guidance sweep (quality-vs-diversity tradeoff).

In [ ]:
!python diversity.py --models sdxl_turbo sd15
!python diversity.py --models sd15 --guidance 2 5 7.5 12
for f in sorted(glob.glob(f"{DATA_DIR}/results/fig[67]*.png")) + sorted(glob.glob(f"{DATA_DIR}/results/contact_*.png")):
    print(os.path.basename(f)); display(Image.open(f))

## 9. Demo app (demo day)
Loads and warms up both models at launch so the first live click is fast. The **Browse** and **Failures & charts** tabs only read saved files, so they still work if the GPU misbehaves on stage. Record the backup video from this app on Saturday night.

In [ ]:
!python demo_app.py --share

In [ ]:
# 10. Package results for the repo / slides
!cd {DATA_DIR} && zip -qr /content/mini-drawbench-results.zip results generations.csv clip_scores.csv ratings.csv
from google.colab import files
files.download("/content/mini-drawbench-results.zip")